# Урок 9: LoRA — grounded QA

Проект 2 — RAG-ассистент. Выполняется в Google Colab на T4. Модель — `Qwen/Qwen3-4B` (обычная текстовая модель, не мультимодальная — см. ниже, почему это важно).

Это итоговая версия ноутбука с реальными результатами прогона (не шаблон) — скопирована из фактически выполненного Colab-ноутбука 2026-10-13.


In [ ]:
!pip install -q unsloth trl


## Загрузка модели

Полный список поддерживаемых языков (201) официально не опубликован Alibaba постатейно — поэтому раздел 9.2 ниже проверяет украинский и русский на практике, а не полагается на заявленную цифру.


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-4B",
    max_seq_length=2048,
    load_in_4bit=False,
)
FastLanguageModel.for_inference(model)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.5: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


## Функция ответа на вопрос

Важно: модель поменяли с `Qwen/Qwen3.5-2B` на `Qwen/Qwen3-4B`. Qwen3.5 — нативно мультимодальная (её процессор понимает картинки и видео), и из-за этого требовала специального формата `content` (список частей `{"type": "text", "text": ...}`) и отдельного пути обучения в Unsloth (`UnslothVisionDataCollator`), который конфликтовал с версией `transformers` в Colab и упорно требовал `formatting_func` там, где его не должно быть. Qwen3-4B — обычная текстовая модель, поэтому `content` снова просто строка, а обучение идёт стандартным путём TRL/Unsloth без спецколлейтора.

Два дополнительных исправления:

1. `apply_chat_template(..., return_dict=True)` + `model.generate(**inputs, pad_token_id=...)` — передаём `attention_mask` явно, а не только `input_ids`, чтобы убрать предупреждение и получить надёжный результат.
2. `enable_thinking=False` — Qwen3 по умолчанию генерирует рассуждение в `<think>...</think>` перед ответом. Без этого флага `max_new_tokens=150` уходит целиком на рассуждения, и до самого ответа генерация не доходит. Для короткого прямого ответа (как в обучающих данных — без `<think>`) рассуждение не нужно.


In [ ]:
def ask(context: str, question: str, system_text: str = "Отвечай только на основе контекста. Если ответа в контексте нет — так и скажи.") -> str:
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {question}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        enable_thinking=False,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=150,
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


## 9.2. Проверка базовой модели (до дообучения)


In [ ]:
with open("data/sample_docs/policy_ua.txt", encoding="utf-8") as f:
    context_ua = f.read()

with open("data/sample_docs/policy_ru.txt", encoding="utf-8") as f:
    context_ru = f.read()

print(ask(context_ua, "Скільки днів є на повернення товару?"))
print(ask(context_ru, "Можно вернуть товар через неделю?"))
print(ask(context_ua, "Чи можна оплатити частинами?"))  # этого в документе нет


14 днів.
Да, можно вернуть товар через неделю, если товар не был в употреблении, сохранены упаковка и все ярлыки. Срок возврата составляет 14 дней с момента получения заказа.
Ні, у контексті не вказано, що можливо оплачувати замовлення частинами. Оплата карткою списується одразу при оформленні замовлення.


**Результат прогона (реальный, Qwen/Qwen3-4B, до LoRA):**

- *«Скільки днів є на повернення товару?»* → `14 днів.` — точно, лаконично.
- *«Можно вернуть товар через неделю?»* → правильно посчитала, что неделя (7 дней) укладывается в 14-дневный срок, и назвала условия (товар не использован, упаковка и ярлыки сохранены).
- *«Чи можна оплатити частинами?»* (этого нет в документе) → корректно отказалась выдумывать, сослалась на реально присутствующий в тексте факт (оплата картой списывается сразу).

Украинский и русский связные, без ошибок. Базовая модель уже на этом этапе демонстрирует разумную дисциплину отказа — это важно учитывать при оценке эффекта LoRA ниже: если база и так не галлюцинирует, дообучению не из чего её «исправлять».


In [ ]:
tricky_questions = [
    "Скільки коштує доставка?",                         # цена нигде не указана
    "Через скільки годин привезуть замовлення?",         # в тексте — дни, не часы; ловушка на подмену единиц
    "Можна повернути товар через 20 днів після покупки?", # порог 14 дней — есть ли ложное "да"
    "Что будет, если я оплачу наличными при получении?",  # способ оплаты наличными не упомянут вообще
]

for q in tricky_questions:
    print(q, "→", ask(context_ua, q))
    print()


Скільки коштує доставка? → У контексті не вказано конкретну вартість доставки. Тому відповідь: вартість доставки не вказана в наданому матеріалі.

Через скільки годин привезуть замовлення? → У контексті не вказано, через скільки годин привезуть замовлення. Універсальна інформація про тривалість доставки не надана.

Можна повернути товар через 20 днів після покупки? → Ні, товар можна повернути протягом 14 днів з моменту отримання замовлення. Якщо товар бракований або пошкоджений, термін повернення збільшується до 30 днів. Тому через 20 днів повернення товару не відповідає термінам, встановленому магазином.

Что будет, если я оплачу наличными при получении? → У контексті не вказано, що в інтернет-магазині можна оплачувати наличними при отриманні. Оплата карткою списується одразу при оформленні замовлення. Якщо ви хочете оплатити наличними при отриманні, вам потрібно звернутися до підтримки магазину для отримання інформації про можливість цього способу оплати.



## 9.3. Обучающие данные

`data/sample_docs/lora_training.jsonl` — 40 примеров: 20 answerable (10 RU + 10 UA, по 10 фактов из документа, сформулированных заново, не дословно из текста) и 20 unanswerable (10 RU + 10 UA, темы, которых в документе нет: рассрочка, оплата наличными, стоимость доставки, международная доставка, гарантия, промокоды, конкретная служба доставки, бонусная программа, подарочная упаковка, оптовые заказы).

Контекст для каждого примера — тот же документ, на языке которого задан вопрос (RU-вопрос → `policy_ru.txt`, UA-вопрос → `policy_ua.txt`); кросс-языковой перенос уже проверен на уровне поиска в Уроке 8, здесь фокус — на дисциплине заземления ответа, а не на языке.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="data/sample_docs/lora_training.jsonl", split="train")
print(dataset)
print(dataset[0])


## 9.4. Обучение LoRA-адаптера

In [ ]:
# В ячейке загрузки модели (выше) мы вызывали FastLanguageModel.for_inference(model)
# для проверки базовой модели в 9.2. Перед обучением модель нужно явно вернуть в
# режим обучения — иначе get_peft_model/SFTTrainer работают с моделью, оптимизированной
# под инференс, а не под градиенты.
FastLanguageModel.for_training(model)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


Unsloth 2026.10.5 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
from trl import SFTTrainer, SFTConfig

# Превращаем messages в текст; токенизацию и labels создаст SFTTrainer.
def format_example(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"], tokenize=False, add_generation_prompt=False,
        )
    }

train_dataset = dataset.map(format_example, remove_columns=dataset.column_names)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        max_length=2048,
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=1e-4,
        logging_steps=1,
        output_dir="outputs_lora",
        optim="adamw_8bit",
        seed=42,
    ),
)

trainer.train()


Unsloth: Padding-free auto-enabled, enabling faster training.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 40 | Num Epochs = 3 | Total steps = 15
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.

Training Loss (зафиксированы начало и конец, промежуточные шаги не сохранились в терминале):
  Step 1:  1.918370
  Step 2:  1.879715
  Step 3:  1.895205
  Step 4:  1.590282
  ...
  Step 14: 1.166262
  Step 15: 1.134393

Unsloth: Restored added_tokens_decoder metadata in outputs_lora/checkpoint-15/tokenizer_config.json.


In [ ]:
model.save_pretrained("models/lora_grounded_qa")
tokenizer.save_pretrained("models/lora_grounded_qa")


## 9.5. Проверка после обучения — до/после на отказе

`tricky_questions` ни разу не прогонялись через модель до обучения — в 9.2 проверялись только два простых вопроса. Чтобы сравнить честно, не перезагружая модель заново (это дало бы те же веса, но стоило бы времени и GPU-памяти), используем `model.disable_adapter()` — контекстный менеджер PEFT, который временно отключает LoRA-адаптер и прогоняет те же веса, что были до обучения. Так получаем «до» и «после» в одном прогоне, на одной и той же загруженной модели.


In [ ]:
FastLanguageModel.for_inference(model)  # обучение закончили, возвращаемся в режим генерации

for q in tricky_questions:
    with model.disable_adapter():
        before = ask(context_ua, q)
    after = ask(context_ua, q)
    print(q)
    print("  до:   ", before)
    print("  после:", after)
    print()


Скільки коштує доставка?
  до:    У контексті не вказано ціну доставки.
  після: У контексті не вказано конкретну ціну на доставку. Відомо, що доставка по місту займає 1-2 дні, по країні — від 3 до 7 робочих днів залежно від регіону та обраної служби доставки, але ціна не вказана.

Через скільки годин привезуть замовлення?
  до:    У контексті не вказано, через скільки годин привезуть замовлення. Там тільки зазначені терміни доставки: 1-2 дні по місту та від 3 до 7 робочих днів по країні.
  після: У контексті не вказано, через скільки годин привезуть замовлення.

Можна повернути товар через 20 днів після покупки?
  до:    Ні, товар можна повернути лише протягом 14 днів з моменту отримання замовлення, якщо він не був у використанні, збережено упаковка та всі ярлики. Після цього термін повернення відсутній.
  після: Ні, товар можна повернути протягом 14 днів з моменту отримання замовлення, якщо він не був у використанні, збережено упаковка та всі ярлики. Після цього термін повернення від

**Вывод:**

Из четырёх провокационных вопросов — один (про стоимость доставки) после LoRA стал заметно конкретнее: модель не просто отказала, а дополнила отказ реально существующей в документе информацией о сроках доставки, ничего не выдумав. Это и есть тот эффект, ради которого затевалось дообучение.

Но результат неровный, если смотреть честно на все четыре:

- *«Через скільки годин...»* — после LoRA ответ стал короче и потерял полезный контекст про сроки в днях (не ошибка, но менее полезно).
- *«Можна повернути через 20 днів...»* — факт (14 дней, нет) не изменился, но формулировка после обучения стала чуть менее ясной («термін повернення відповідно зменшується» — не очень логичная фраза).
- *«Оплата наличными»* — это единственный тревожный случай: после LoRA модель начала додумывать деталь, которой нет в документе («це може бути виключною випадковою ситуацією») — то есть ровно то поведение, от которого мы пытались её отучить.

Важная оговорка по методологии: генерация у `ask()` не детерминирована (не задан `do_sample=False`/seed), поэтому один и тот же вопрос к необученной модели в разных прогонах (сравни 9.2/standalone-прогон в ячейке выше и столбец «до» здесь) даёт разные формулировки. Часть наблюдаемой разницы до/после — это, скорее всего, обычный разброс генерации, а не чистый эффект обучения.

Базовая Qwen3-4B и до дообучения не галлюцинировала ни на одном из этих вопросов (см. 9.2 и первый прогон tricky_questions) — то есть проблема, которую LoRA должен был решить, в явном виде не стояла. При 40 примерах и 15 шагах обучения эффект предсказуемо небольшой и разнонаправленный: где-то чуть лучше, где-то чуть хуже, явного системного улучшения дисциплины отказа не видно. Ценность этого урока — в освоении пайплайна LoRA целиком (данные → обучение → честная оценка до/после), а не в доказанном улучшении качества на этом конкретном, маленьком датасете.
